# 1. Retrieval, built from nothing

Working through how the knowledge base actually finds things, starting from an empty file. No imports from the project until the very end, where I check what I built against `app/kb/loader.py`.

The thing I want to understand: why does an off-topic message need to return an empty list, and what breaks if it doesn't.


## The problem

A prospect texts something. I need to decide which product facts, if any, the agent is allowed to use when replying. Get this wrong in one direction and the agent invents things. Wrong in the other and it refuses questions it could easily answer.

Start with the smallest thing that could work: a dict.


In [3]:
FACTS = {
    'what is voicecaptures': 'An AI voice receptionist for home service businesses.',
    'is this a robot': "Yes, it's an AI voice agent, not a recording.",
    'free trial': "There's a 14-day free trial.",
}

def lookup_v1(message):
    return FACTS.get(message)

print(lookup_v1('what is voicecaptures'))
print(lookup_v1('What is VoiceCaptures?'))   # already broken


An AI voice receptionist for home service businesses.
None


Exact-match dies immediately. Real messages have capitals, punctuation,
and never match a key character for character.

## Attempt 2: substring


In [7]:
def lookup_v2(message):
    low = message.lower()
    for key, answer in FACTS.items():
        if key in low:
            return answer
    return None

for m in ['What is VoiceCaptures?', 'is this a robot lol', 'do you have a free trial', 
          'is there trial that is free']:
    print(f'{m!r:32} -> {lookup_v2(m)}')


'What is VoiceCaptures?'         -> An AI voice receptionist for home service businesses.
'is this a robot lol'            -> Yes, it's an AI voice agent, not a recording.
'do you have a free trial'       -> There's a 14-day free trial.
'is there trial that is free'    -> None


Better. But one key per fact is thin — a prospect can phrase the robot question ten ways. So: several trigger phrases per entry.

## Attempt 3: multiple triggers


In [8]:
ENTRIES = [
    {'id': 'what_is_voicecaptures',
     'triggers': ['what is voicecaptures', 'what do you do', 'what is this'],
     'answer': 'An AI voice receptionist for home service businesses.'},
    {'id': 'is_it_a_robot',
     'triggers': ['is this a bot', 'is it a robot', 'are you a real person', 'robotic'],
     'answer': "Yes, it's an AI voice agent, not a recording."},
    {'id': 'free_trial',
     'triggers': ['free trial', 'try it', 'trial'],
     'answer': "There's a 14-day free trial."},
]

def lookup_v3(message):
    low = message.lower()
    for e in ENTRIES:
        if any(t in low for t in e['triggers']):
            return e['id']
    return None

for m in ['what do you do exactly', 'is it a robot?', 'any trial?']:
    print(f'{m!r:26} -> {lookup_v3(m)}')


'what do you do exactly'   -> what_is_voicecaptures
'is it a robot?'           -> is_it_a_robot
'any trial?'               -> free_trial


## Where this breaks

This is the version that shipped, and it caused a real problem. A (rather funny!) prospect texted 'You like feet?' and got a reply about pricing.

`'fee' in 'do you like feet'` is True. Substring matching doesn't care about word boundaries.


In [9]:
RESTRICTED = [
    {'id': 'pricing', 'triggers': ['price', 'cost', 'how much', 'fee', 'rate', 'quote']},
    {'id': 'contract_terms', 'triggers': ['contract', 'cancel', 'refund']},
]

def restricted_v1(message):
    low = message.lower()
    for r in RESTRICTED:
        if any(t in low for t in r['triggers']):
            return r['id']
    return None

for m in ['do you like feet', 'that quote was accurate', 'he wore a costume',
          'we need a rapid response', 'how much does it cost']:
    print(f'{m!r:28} -> {restricted_v1(m)}')


'do you like feet'           -> pricing
'that quote was accurate'    -> pricing
'he wore a costume'          -> pricing
'we need a rapid response'   -> None
'how much does it cost'      -> pricing


Three false positives out of five. 'fee' inside 'feet', 'rate' inside 'accurate', 'cost' inside 'costume'.

(Note 'that quote was accurate' — 'quote' really IS a pricing trigger, so that one is arguably correct. Took me a minute to notice.)

## The fix: word boundaries

`\b` in regex means 'edge of a word'. It handles multi-word phrases too, so 'how much' still works.


In [13]:
import re

def phrase_in(phrase, text):
    return re.search(rf'\b{re.escape(phrase)}\b', text) is not None

def restricted_v2(message):
    low = message.lower()
    for r in RESTRICTED:
        if any(phrase_in(t, low) for t in r['triggers']):
            return r['id']
    return None

print(f'{"message":30} {"substring":16} word-boundary')
for m in ['do you like feet', 'he wore a costume',  
          'how much does it cost', 'is there a contract', 
          'is there contraction?', 'we need a rapid response']:
    print(f'{m:30} {str(restricted_v1(m)):16} {restricted_v2(m)}')


message                        substring        word-boundary
do you like feet               pricing          None
he wore a costume              pricing          None
how much does it cost          pricing          pricing
is there a contract            contract_terms   contract_terms
is there contraction?          contract_terms   None
we need a rapid response       None             None


## Paraphrase: the thing triggers can't cover

Literal phrases only fire when the prospect uses my words. 'How does this work' shares no vocabulary with any trigger I wrote.

So: score on overlapping words as well, not just literal phrases. Strip the filler words first or everything matches everything.


In [14]:
# Words that appear frequently but usually do not add much meaning when
# matching user queries. Removing these helps keep only the important keywords.
STOPWORDS = {
    'a','an','and','are','as','at','be','but','by','can','do','does',
    'for','from','get','has','have','how','i','if','in','is','it','its',
    'me','my','no','not','of','on','or','so','that','the','their','them',
    'then','there','they','this','to','up','us','was','we','what','when',
    'who','will','with','you','your','yes','ok','okay'
}


# Extract meaningful words from a text input.
# The text is normalized to lowercase, split into individual words,
# and filtered to remove common words and very short tokens.
def tokens_v1(text):
    words = re.findall(r"[a-z']+", text.lower())
    return {w for w in words if w not in STOPWORDS and len(w) > 2}


# Try the tokenizer on a few example user messages to see
# which keywords are extracted from each query.
for m in [
    'how does this work',
    'is this a robot',
    "what's the weather in the uk"
]:
    print(f'{m!r:32} -> {sorted(tokens_v1(m))}')

'how does this work'             -> ['work']
'is this a robot'                -> ['robot']
"what's the weather in the uk"   -> ['weather', "what's"]


Look at the third one. `what's` survived.

The stopword list has `what`, but the regex kept the apostrophe, so the token is `what's` and never matches. It sails through as a content word.

That's not cosmetic --watch what it does:


In [21]:
ENTRIES[0]['triggers']

['what is voicecaptures',
 'what do you do',
 'what is this',
 "what's this about",
 "what's this about",
 "what's this about",
 "what's this about",
 "what's this about"]

In [20]:
def score_v1(message, entry):
    # calculate how closely a user message matches a knowledge base entry
    # Higher scores mean the entry is more likely to be relevant
    low = message.lower()
    s = 0.0

    # Give a strong score boost when the user's message contains
    # one of the predefined trigger phrases for this entry.
    for t in entry['triggers']:
        if phrase_in(t, low):
            s += 10.0

    # Ccompare the keywords in the user's message with the keywords
    # from the entry's triggers. Shared keywords provide an additional
    # smaller similarity score.
    trigger_tokens = tokens_v1(' '.join(entry['triggers']))
    s += 2.0 * len(tokens_v1(message) & trigger_tokens)

    return s


# Add a question-style trigger to simulate how a real knowledge base
# might include different ways users ask about the same topic.
ENTRIES[0]['triggers'].append("what's this about")


# Test the scoring function by comparing a user query against
# each knowledge base entry and displaying the resulting scores.
q = "what's the weather like in the uk"

for e in ENTRIES:
    print(f"{e['id']:24} score={score_v1(q, e)}")

what_is_voicecaptures    score=2.0
is_it_a_robot            score=0.0
free_trial               score=0.0


An off-topic question about British weather retrieves the product overview, because both contain `what's`.

I had tested this by hand and it passed; I typed it without the spostrophe.

Fix: strip apostrophes before checking stopwords, and add the contraction forms to the list.


In [22]:
STOPWORDS |= {'whats','thats','hows','wheres','whos','im','ive','dont',
              'doesnt','isnt','cant','wont','youre','its','lets'} # |= is the in-place union operator for sets, similar to += which is in-place addition

def tokens(text):
    words = re.findall(r"[a-z']+", text.lower())
    cleaned = (w.replace("'", '') for w in words)
    return {w for w in cleaned if w not in STOPWORDS and len(w) > 2}

print('old:', sorted(tokens_v1(q)))
print('new:', sorted(tokens(q)))


old: ['like', 'weather', "what's"]
new: ['like', 'weather']


## Putting it together


In [24]:
MIN_SCORE = 2.0

def score(message, entry):
    low = message.lower()
    s = 0.0
    for t in entry['triggers']:
        if phrase_in(t, low):
            s += 10.0 + len(t) / 10.0
    s += 2.0 * len(tokens(message) & tokens(' '.join(entry['triggers'])))
    return s

def search(message, limit=3):
    scored = [(score(message, e), e) for e in ENTRIES]
    hits = [(s, e) for s, e in scored if s >= MIN_SCORE]
    hits.sort(key=lambda p: (-p[0], p[1]['id']))
    return [e for _, e in hits[:limit]]

for m in ['what is voicecaptures', 'is this a robot', 'do you have a free trial',
          "do you like feet", "what's the weather like in the uk", 'who won the game', 
          'lol']:
    print(f'{m!r:36} -> {[e["id"] for e in search(m)] or "nothing"}')


'what is voicecaptures'              -> ['what_is_voicecaptures']
'is this a robot'                    -> ['is_it_a_robot']
'do you have a free trial'           -> ['free_trial']
'do you like feet'                   -> nothing
"what's the weather like in the uk"  -> nothing
'who won the game'                   -> nothing
'lol'                                -> nothing


## Why empty matters

The last four return `[]`, and that empty list is the whole point.

The agent's instruction willbe: state facts only from the entries provided. No entries means no facts may be stated. It's not a soft hint, it's the absence of raw material.

A vector store cannot give me that. Cosine similarity returns a nearest neighbour for every query; ask about the weather and you get whichever
product fact is least unrelated, with some score like 0.31. To reject it I'd pick a threshold, and then I'd own that threshold forever and have to defend the number or constantly adjust it.

Here, 'no trigger matched and no content word overlapped' means nothing matched. No tuning.

The tradeoff is important: this misses paraphrase. 'How does this work' returns nothing above, and that's the most common question a prospect can ask. I patched it by adding trigger phrases, which works and doesn't generalise (see later notebooks). Open question I haven't answered: whether to keep patching or add embeddings as a fallback for the empty case.

## What this became

Everything above is `app/kb/loader.py`. Same functions, different names:

| here | in the module |
|---|---|
| `phrase_in` | `_phrase_in` |
| `tokens` | `_tokens` |
| `score` + `search` | `search` |
| `restricted_v2` | `match_restricted` |
| `ENTRIES` list | `voicecaptures.yaml`, parsed into `Entry` dataclasses |
| `MIN_SCORE` | `MIN_RETRIEVAL_SCORE` |

What the module adds that I skipped: YAML loading so non-code people can edit facts, `lru_cache` so the file is parsed once, validation that raises at import rather than at 2am, and dataclasses instead of dicts.

Check they agree:


In [25]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else '.')
for k, v in {'SUPABASE_URL':'https://t.supabase.co','SUPABASE_SERVICE_KEY':'t',
             'OPENAI_API_KEY':'sk-t','TWILIO_ACCOUNT_SID':'ACt',
             'TWILIO_AUTH_TOKEN':'t','TWILIO_FROM_NUMBER':'+15550000000'}.items():
    os.environ.setdefault(k, v)

from app.kb.loader import search as real_search, match_restricted as real_restricted

for m in ["what's the weather like in the uk", 'do you like feet',
          'how much does it cost', 'is this a robot']:
    r = real_restricted(m)
    hits = [e.id for e in real_search(m)]
    tag = r.id if r else '-'
    print(f'{m!r:36} restricted={tag:<12} retrieved={hits or "nothing"}')


ModuleNotFoundError: No module named 'app'

## Aside: normalizing the phone number

Not retrieval, but it's the other pure function in this system and it broke in a way that's worth sitting with.

Every number entering the system has to become of this format -- `+14168226186` -- because that's what Twilio accepts. Leads arrive from a scraped CSV in whatever format the listing had.


In [27]:
import re


def normalize_v1(raw):
    # Remove any characters that are not digits or the '+' sign.
    # The regex pattern '[^\d+]' means:
    # - [] defines a set of characters to match
    # - ^ inside [] means "anything except"
    # - \d means any digit (0-9)
    # - + is kept because it is used for international prefixes
    digits = re.sub(r'[^\d+]', '', raw)

    # If the number does not already include a country code,
    # assume it is a North American number and add the +1 prefix.
    if not digits.startswith('+'):
        digits = '+1' + digits

    return digits


# Test different phone number formats to confirm they are
# converted into a consistent international format.
for raw in [
    '(416) 822-6186',
    '416-822-6186',
    '4168226186',
    '+14168226186'
]:
    print(f'{raw!r:20} -> {normalize_v1(raw)}')

'(416) 822-6186'     -> +14168226186
'416-822-6186'       -> +14168226186
'4168226186'         -> +14168226186
'+14168226186'       -> +14168226186


Looks fine. Ships. Then a send fails with Twilio error 21211,
`Invalid 'To' Phone Number: +4168226186`.


In [28]:
print(normalize_v1('+416 822 6186'))


+4168226186


A Toronto number typed with a plus but no country code. My check was 'does it start with +', which I'd treated as 'is it already complete
E.164'. Those aren't the same question.

Twilio reads the leading `4` as a country code and rejects it.
 
The 500 at send time was the lucky part. The real damage was at CSV import, where it silently stored corrupted numbers that wouldn't fail
until someone tried to text them.

The fix is to stop asking about the `+` and ask about the shape:


In [33]:
class InvalidPhoneError(ValueError):
    pass

def normalize_v2(raw):
    if raw is None:
        raise InvalidPhoneError('empty')
    digits = re.sub(r'\D', '', str(raw))      # drop the + entirely, count digits
    if not digits:
        raise InvalidPhoneError(str(raw))
    if len(digits) == 10:                     # NANP without country code
        return '+1' + digits
    if len(digits) == 11 and digits.startswith('1'):
        return '+' + digits
    if 7 <= len(digits) <= 15:                # assume it has one already
        return '+' + digits
    raise InvalidPhoneError(str(raw))

for raw in ['+416 822 6186', '(416) 822-6186', '4168226186', '1-416-822-6186',
            '+14168226186', '+44 20 7946 0958', '', 'abc','1111111111111111111111']:
    try:
        print(f'{raw!r:22} -> {normalize_v2(raw)}')
    except InvalidPhoneError:
        print(f'{raw!r:22} -> rejected')


'+416 822 6186'        -> +14168226186
'(416) 822-6186'       -> +14168226186
'4168226186'           -> +14168226186
'1-416-822-6186'       -> +14168226186
'+14168226186'         -> +14168226186
'+44 20 7946 0958'     -> +442079460958
''                     -> rejected
'abc'                  -> rejected
'1111111111111111111111' -> rejected


Raising instead of returning something is the other half of the fix. The old version always produced a string, so bad input became a broken
prospect that failed weeks later at send time, far from the cause.

### The bug I only found by writing a test

I added a case for `555-1234` expecting a rejection, and it passed instead:


In [34]:
print(normalize_v2('555-1234'))


+5551234


Seven digits fell into my '7 to 15, assume it has a country code' branch. But a bare seven-digit string is a local number missing its
area code -- there's no country code in there at all.

`+5551234` is wores than a crash, because it looks plausible.

The shortest real E.164 numbers run about 8 digits, so the floor was simply wrong.


In [35]:
def normalize(raw):
    if raw is None:
        raise InvalidPhoneError('empty')
    digits = re.sub(r'\D', '', str(raw))
    if not digits:
        raise InvalidPhoneError(str(raw))
    if len(digits) == 10:
        return '+1' + digits
    if len(digits) == 11 and digits.startswith('1'):
        return '+' + digits
    if 8 <= len(digits) <= 15:            # was 7
        return '+' + digits
    raise InvalidPhoneError(str(raw))

for raw in ['555-1234', '+44 20 7946 0958', '+416 822 6186']:
    try:
        print(f'{raw!r:22} -> {normalize(raw)}')
    except InvalidPhoneError:
        print(f'{raw!r:22} -> rejected')


'555-1234'             -> rejected
'+44 20 7946 0958'     -> +442079460958
'+416 822 6186'        -> +14168226186


### One more property worth asserting

Numbers get re-normalized when a prospect is edited, so a second pass has to be a no-op. A version that prepended `+1` every time would pass
every test above and still corrupt on the second edit.


In [40]:
# Check that normalize() is idempotent.
# This means running the function multiple times should not
# keep changing the output after the first normalization.
#
# For example:
# normalize("416-822-6186") -> "+14168226186"
# normalize("+14168226186") -> "+14168226186"

for raw in ['+416 822 6186', '4168226186', 
            '+44 20 7946 0958']:
    once = normalize(raw)

    # Verify that normalizing an already-normalized value
    # produces the same result. If not, raise an error.
    assert normalize(once) == once, f'not idempotent: {raw}'

print('idempotent')

idempotent


### Where it lives

This is `app/phone.py`, and the location is itself a lesson.

It originally sat in `app/db/import_csv.py`, which imports the Supabase client. So testing a regex required a database driver and live
credentials -- which meant the fast test tier wasn't fast, and I noticed only when I tried to run the tests somewhere without a database.

It's now its own module importing nothing but `re`. `import_csv` re-exports it so existing callers didn't change.

The general shape: **a pure function guarded by fast tests shouldn't drag in infrastructure.** If it does, the tests stop being cheap and
people stop running them.

Check against the real thing:


In [ ]:
from app.phone import normalize_phone, InvalidPhoneError as RealError

for raw in ['+416 822 6186', '555-1234', '+44 20 7946 0958', '4168226186']:
    try:
        print(f'{raw!r:22} -> {normalize_phone(raw)}')
    except RealError:
        print(f'{raw!r:22} -> rejected')


Both of these -- the `+` assumption and the 7-digit floor -- are in `tests/test_phone_normalization.py` now, so they can't come back.

Next notebook: what happens to a message before retrieval runs, and what checks the reply afterwards.
